# 06 — Data Export for Dashboard

Prepares final aggregated tables for Tableau. Pre-computes all metrics so Tableau visualizes, not computes.

Requires `cms_data.db` from `01_setup.ipynb` and analysis from `03-05`.

In [12]:
import sqlite3
import pandas as pd
from pathlib import Path

In [13]:
DB = Path("../cms_data.db")
EXPORT = Path("../data/exports")
conn = sqlite3.connect(DB)

---
## 1. DRG-level summary (cost + readmission + utilization)

In [14]:
drg_summary = pd.read_sql("""
WITH drg_costs AS (
  SELECT
    CLM_DRG_CD,
    COUNT(*) as num_claims,
    ROUND(SUM(CLM_PMT_AMT), 0) as total_cost,
    ROUND(AVG(CLM_PMT_AMT), 0) as avg_cost_per_claim,
    ROUND(AVG(CLM_UTLZTN_DAY_CNT), 1) as avg_los
  FROM inpatient_claims
  WHERE CLM_PMT_AMT > 0
  GROUP BY CLM_DRG_CD
),
drg_readmit AS (
  SELECT
    CLM_DRG_CD,
    SUM(CASE WHEN days_to_readmission > 0 AND days_to_readmission <= 30 THEN 1 ELSE 0 END) as readmissions_30d,
    COUNT(*) as total_admits
  FROM (
    SELECT
      CLM_DRG_CD,
      CAST((julianday(next_admsn) - julianday(dschg_dt)) AS INTEGER) as days_to_readmission
    FROM (
      SELECT
        CLM_DRG_CD,
        NCH_BENE_DSCHRG_DT as dschg_dt,
        LEAD(CLM_ADMSN_DT) OVER (PARTITION BY DESYNPUF_ID ORDER BY CLM_ADMSN_DT) as next_admsn
      FROM inpatient_claims
    )
    WHERE next_admsn IS NOT NULL
  )
  GROUP BY CLM_DRG_CD
)
SELECT
  c.CLM_DRG_CD as drg,
  c.num_claims,
  c.total_cost,
  c.avg_cost_per_claim,
  c.avg_los,
  COALESCE(r.readmissions_30d, 0) as readmissions_30d,
  ROUND(COALESCE(r.readmissions_30d, 0) * 100.0 / COALESCE(r.total_admits, 1), 1) as readmission_rate_pct
FROM drg_costs c
LEFT JOIN drg_readmit r ON c.CLM_DRG_CD = r.CLM_DRG_CD
ORDER BY c.total_cost DESC
""", conn)

drg_summary.to_csv(EXPORT / "drg_summary.csv", index=False)
print(f"DRG summary: {len(drg_summary)} rows")
print(drg_summary.head(10))

DRG summary: 739 rows
   drg  num_claims  total_cost  avg_cost_per_claim  avg_los  readmissions_30d  \
0  939         249   3516800.0             14124.0     11.4                 0   
1  948         231   3411700.0             14769.0     10.6                 0   
2  949         233   3328000.0             14283.0     11.3                 0   
3  945         237   3318000.0             14000.0     11.7                 0   
4  951         236   3275000.0             13877.0     10.8                 0   
5  946         214   3071100.0             14351.0     11.1                 0   
6  941         216   2982600.0             13808.0     10.7                 0   
7  950         203   2859900.0             14088.0     11.2                 0   
8  864         190   2852000.0             15011.0      7.3                 0   
9  862         186   2845530.0             15299.0      8.0                 0   

   readmission_rate_pct  
0                   0.0  
1                   0.0  
2       

---
## 2. Diagnosis-level summary (cost + volume)

In [15]:
diagnosis_summary = pd.read_sql("""
SELECT
  ICD9_DGNS_CD_1 as diagnosis,
  COUNT(*) as num_claims,
  ROUND(SUM(CLM_PMT_AMT), 0) as total_cost,
  ROUND(AVG(CLM_PMT_AMT), 0) as avg_cost_per_claim,
  COUNT(DISTINCT DESYNPUF_ID) as unique_beneficiaries
FROM inpatient_claims
WHERE CLM_PMT_AMT > 0 AND ICD9_DGNS_CD_1 IS NOT NULL
GROUP BY ICD9_DGNS_CD_1
ORDER BY total_cost DESC
""", conn)

diagnosis_summary.to_csv(EXPORT / "diagnosis_summary.csv", index=False)
print(f"Diagnosis summary: {len(diagnosis_summary)} rows")
print(diagnosis_summary.head(10))

Diagnosis summary: 2699 rows
  diagnosis  num_claims  total_cost  avg_cost_per_claim  unique_beneficiaries
0     V5789        1772  28421600.0             16039.0                  1712
1      0389        1602  24410390.0             15237.0                  1560
2     41401        1596  23847800.0             14942.0                  1560
3       486        2394  17101860.0              7144.0                  2312
4     41071        1124  14809630.0             13176.0                  1091
5     51881         795  14232900.0             17903.0                   777
6     71536        1101  13133020.0             11928.0                  1081
7      4280        1378  11069900.0              8033.0                  1341
8     49121        1520   9999300.0              6578.0                  1455
9      5849        1064   8624900.0              8106.0                  1038


---
## 3. Beneficiary cohort segmentation (spending tier + condition count)

In [16]:
bene = pd.read_sql("""
SELECT
  DESYNPUF_ID,
  YEAR,
  BENE_BIRTH_DT,
  BENE_DEATH_DT,
  BENE_SEX_IDENT_CD,
  COALESCE(MEDREIMB_IP, 0) + COALESCE(MEDREIMB_OP, 0) + COALESCE(MEDREIMB_CAR, 0) as total_spend,
  MEDREIMB_IP,
  MEDREIMB_OP,
  SP_DIABETES,
  SP_CHF,
  SP_COPD,
  SP_ISCHMCHT,
  SP_DEPRESSN,
  SP_ALZHDMTA,
  SP_CHRNKIDN,
  SP_OSTEOPRS,
  SP_RA_OA,
  SP_STRKETIA,
  SP_CNCR
FROM beneficiary_summary
""", conn)

chronic_cols = ['SP_DIABETES', 'SP_CHF', 'SP_COPD', 'SP_ISCHMCHT', 'SP_DEPRESSN',
                'SP_ALZHDMTA', 'SP_CHRNKIDN', 'SP_OSTEOPRS', 'SP_RA_OA', 'SP_STRKETIA', 'SP_CNCR']

bene['num_conditions'] = (bene[chronic_cols] == 1).sum(axis=1)

# Segment into spending tiers
bene['spending_tier'] = pd.cut(bene['total_spend'],
                                bins=[0, 500, 2000, 5000, 10000, 250000],
                                labels=['$0-500', '$500-2K', '$2K-5K', '$5K-10K', '$10K+'])

beneficiary_cohort = bene.copy()
beneficiary_cohort.to_csv(EXPORT / "beneficiary_cohort.csv", index=False)
print(f"Beneficiary cohort: {len(beneficiary_cohort)} rows")
print(f"\nSpending tier distribution:")
print(beneficiary_cohort['spending_tier'].value_counts().sort_index())

Beneficiary cohort: 343644 rows

Spending tier distribution:
spending_tier
$0-500     51817
$500-2K    94066
$2K-5K     63123
$5K-10K    26570
$10K+      30887
Name: count, dtype: int64


---
## 4. Chronic condition impact summary

In [17]:
condition_impact = pd.DataFrame()

for condition in chronic_cols:
    with_cond = bene[bene[condition] == 1]
    without_cond = bene[bene[condition] == 2]
    
    row = {
        'condition': condition.replace('SP_', ''),
        'beneficiary_years_with_condition': len(with_cond),
        'prevalence_pct': round(len(with_cond) / len(bene) * 100, 1),
        'avg_spend_with': round(with_cond['total_spend'].mean()),
        'avg_spend_without': round(without_cond['total_spend'].mean()),
        'median_spend_with': round(with_cond['total_spend'].median()),
        'median_spend_without': round(without_cond['total_spend'].median())
    }
    row['spend_delta'] = row['avg_spend_with'] - row['avg_spend_without']
    condition_impact = pd.concat([condition_impact, pd.DataFrame([row])], ignore_index=True)

condition_impact = condition_impact.sort_values('spend_delta', ascending=False).reset_index(drop=True)
condition_impact.to_csv(EXPORT / "condition_impact.csv", index=False)
print("Chronic condition impact summary:")
print(condition_impact)

Chronic condition impact summary:
   condition  beneficiary_years_with_condition  prevalence_pct  \
0   STRKETIA                             14204             4.1   
1   CHRNKIDN                             58023            16.9   
2       COPD                             43773            12.7   
3       CNCR                             22484             6.5   
4        CHF                            101752            29.6   
5   DIABETES                            124747            36.3   
6   ISCHMCHT                            145708            42.4   
7   ALZHDMTA                             67503            19.6   
8      RA_OA                             48822            14.2   
9   DEPRESSN                             72967            21.2   
10  OSTEOPRS                             56776            16.5   

    avg_spend_with  avg_spend_without  median_spend_with  \
0            13181               3201               6380   
1            11481               2015               5

---
## 5. Comorbidity pairs (high-risk combinations)

In [18]:
comorbidity_pairs = pd.DataFrame()

pairs = [
    ('SP_DIABETES', 'SP_CHF'),
    ('SP_DIABETES', 'SP_ISCHMCHT'),
    ('SP_CHF', 'SP_COPD'),
    ('SP_ISCHMCHT', 'SP_DEPRESSN'),
    ('SP_CHRNKIDN', 'SP_DIABETES'),
    ('SP_CHF', 'SP_ISCHMCHT'),
    ('SP_COPD', 'SP_DEPRESSN'),
]

for cond1, cond2 in pairs:
    both = bene[(bene[cond1] == 1) & (bene[cond2] == 1)]
    either = bene[((bene[cond1] == 1) | (bene[cond2] == 1))]
    neither = bene[(bene[cond1] == 2) & (bene[cond2] == 2)]
    
    row = {
        'condition_1': cond1.replace('SP_', ''),
        'condition_2': cond2.replace('SP_', ''),
        'count_both': len(both),
        'prevalence_both_pct': round(len(both) / len(bene) * 100, 1),
        'avg_spend_both': round(both['total_spend'].mean()),
        'avg_spend_either': round(either['total_spend'].mean()),
        'avg_spend_neither': round(neither['total_spend'].mean())
    }
    comorbidity_pairs = pd.concat([comorbidity_pairs, pd.DataFrame([row])], ignore_index=True)

comorbidity_pairs = comorbidity_pairs.sort_values('avg_spend_both', ascending=False).reset_index(drop=True)
comorbidity_pairs.to_csv(EXPORT / "comorbidity_pairs.csv", index=False)
print("High-risk comorbidity combinations:")
print(comorbidity_pairs)

High-risk comorbidity combinations:
  condition_1 condition_2  count_both  prevalence_both_pct  avg_spend_both  \
0         CHF        COPD       30287                  8.8           13350   
1        COPD    DEPRESSN       20246                  5.9           13348   
2    CHRNKIDN    DIABETES       45570                 13.3           12495   
3    DIABETES         CHF       68580                 20.0            9810   
4         CHF    ISCHMCHT       77020                 22.4            9407   
5    ISCHMCHT    DEPRESSN       51337                 14.9            9286   
6    DIABETES    ISCHMCHT       91642                 26.7            8622   

   avg_spend_either  avg_spend_neither  
0              7859               1472  
1              7978               1910  
2              7337               1139  
3              6683               1003  
4              6409                863  
5              6395                974  
6              6253                750  


---
## 6. Spending and readmission trends by year

In [19]:
yearly_summary = pd.read_sql("""
SELECT
  YEAR,
  COUNT(DISTINCT DESYNPUF_ID) as num_beneficiaries,
  ROUND(AVG(MEDREIMB_IP), 0) as avg_inpatient_spend,
  ROUND(AVG(MEDREIMB_OP), 0) as avg_outpatient_spend,
  ROUND(AVG(COALESCE(MEDREIMB_IP, 0) + COALESCE(MEDREIMB_OP, 0) + COALESCE(MEDREIMB_CAR, 0)), 0) as avg_total_spend
FROM beneficiary_summary
GROUP BY YEAR
ORDER BY YEAR
""", conn)

yearly_summary.to_csv(EXPORT / "yearly_trends.csv", index=False)
print("Spending trends by year:")
print(yearly_summary)

Spending trends by year:
   YEAR  num_beneficiaries  avg_inpatient_spend  avg_outpatient_spend  \
0  2008             116352               2214.0                 622.0   
1  2009             114538               2190.0                 770.0   
2  2010             112754               1242.0                 432.0   

   avg_total_spend  
0           3999.0  
1           4297.0  
2           2522.0  


---
## 7. Readmission summary by DRG

In [20]:
readmission_by_drg = pd.read_sql("""
WITH readmit_flagged AS (
  SELECT
    DESYNPUF_ID,
    CLM_ADMSN_DT,
    NCH_BENE_DSCHRG_DT,
    CLM_DRG_CD,
    CLM_PMT_AMT,
    CAST((julianday(LEAD(CLM_ADMSN_DT) OVER (PARTITION BY DESYNPUF_ID ORDER BY CLM_ADMSN_DT)) - julianday(NCH_BENE_DSCHRG_DT)) AS INTEGER) as days_to_next_admit
  FROM inpatient_claims
  WHERE CLM_ADMSN_DT IS NOT NULL
)
SELECT
  CLM_DRG_CD as drg,
  COUNT(*) as num_admits,
  SUM(CASE WHEN days_to_next_admit > 0 AND days_to_next_admit <= 30 THEN 1 ELSE 0 END) as readmit_30d_count,
  ROUND(SUM(CASE WHEN days_to_next_admit > 0 AND days_to_next_admit <= 30 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as readmit_30d_rate_pct,
  ROUND(AVG(CLM_PMT_AMT), 0) as avg_cost_per_admit
FROM readmit_flagged
GROUP BY CLM_DRG_CD
ORDER BY readmit_30d_rate_pct DESC
""", conn)

readmission_by_drg.to_csv(EXPORT / "readmission_by_drg.csv", index=False)
print(f"Readmission by DRG: {len(readmission_by_drg)} DRGs")
print(readmission_by_drg.head(15))

Readmission by DRG: 739 DRGs
    drg  num_admits  readmit_30d_count  readmit_30d_rate_pct  \
0   OTH         258                  0                   0.0   
1   999          38                  0                   0.0   
2   998          28                  0                   0.0   
3   989          27                  0                   0.0   
4   988          42                  0                   0.0   
5   987          28                  0                   0.0   
6   986          35                  0                   0.0   
7   985          36                  0                   0.0   
8   984          41                  0                   0.0   
9   983          62                  0                   0.0   
10  982          25                  0                   0.0   
11  981          43                  0                   0.0   
12  977          14                  0                   0.0   
13  976          11                  0                   0.0   
14  975    

---
## 8. Summary statistics

In [21]:
summary_stats = pd.DataFrame([
    {
        'metric': 'Total beneficiary-years',
        'value': len(bene)
    },
    {
        'metric': 'Unique beneficiaries',
        'value': bene['DESYNPUF_ID'].nunique()
    },
    {
        'metric': 'Total inpatient claims',
        'value': len(pd.read_sql("SELECT COUNT(*) as cnt FROM inpatient_claims", conn))
    },
    {
        'metric': 'Total outpatient claims',
        'value': len(pd.read_sql("SELECT COUNT(*) as cnt FROM outpatient_claims", conn))
    },
    {
        'metric': 'Total prescription events',
        'value': len(pd.read_sql("SELECT COUNT(*) as cnt FROM prescription_drug_events", conn))
    },
    {
        'metric': 'Total Medicare spend (billions)',
        'value': round(bene['total_spend'].sum() / 1e9, 2)
    },
    {
        'metric': 'Avg annual spend per beneficiary',
        'value': round(bene['total_spend'].mean())
    },
    {
        'metric': 'Avg inpatient readmission rate',
        'value': round(readmission_by_drg['readmit_30d_count'].sum() / readmission_by_drg['num_admits'].sum() * 100, 1)
    }
])

summary_stats.to_csv(EXPORT / "summary_statistics.csv", index=False)
print("\nProject Summary Statistics:")
print(summary_stats.to_string(index=False))


Project Summary Statistics:
                          metric     value
         Total beneficiary-years 343644.00
            Unique beneficiaries 116352.00
          Total inpatient claims      1.00
         Total outpatient claims      1.00
       Total prescription events      1.00
 Total Medicare spend (billions)      1.24
Avg annual spend per beneficiary   3614.00
  Avg inpatient readmission rate      0.00


In [22]:
conn.close()
print("\n" + "="*70)
print("Export complete. All files written to data/exports/")
print("="*70)
print("\nFiles ready for Tableau:")
import os
for f in sorted(os.listdir(EXPORT)):
    if f.endswith('.csv'):
        fsize = os.path.getsize(EXPORT / f) / 1024
        print(f"  {f:<40} {fsize:>8.1f} KB")


Export complete. All files written to data/exports/

Files ready for Tableau:
  beneficiary_cohort.csv                    27120.5 KB
  comorbidity_pairs.csv                         0.4 KB
  condition_impact.csv                          0.6 KB
  condition_progression_by_year.csv             0.4 KB
  diagnosis_summary.csv                        70.1 KB
  drg_summary.csv                              25.3 KB
  dual_condition_combinations.csv               0.3 KB
  readmission_by_chronic_condition.csv          0.2 KB
  readmission_by_drg.csv                       15.7 KB
  readmission_rates_by_diagnosis.csv           40.8 KB
  readmission_rates_by_drg.csv                 10.9 KB
  spending_by_condition_burden.csv              0.5 KB
  summary_statistics.csv                        0.3 KB
  yearly_trends.csv                             0.2 KB
